Загрузим таблицы

In [2]:
import pandas as pd

deals_df = pd.read_excel("Deals_f_clean.xlsx", dtype={"Id": str,"Contact Name": str})
spend_df = pd.read_excel("Spend_f_clean.xlsx")


**Посчитаем юнит-экономику для всех продуктов. За весь период (год)**

*   UA -  количество уникальных потенциальных клиентов (взяла их из таблицы Deals по полю Contact Name). Это пользователи, которые оставили заявки на сайте. В таблице Contacts их немного больше, но так как хочу сделать воронку, решила взять именно этих UA.
*   B - количество уникальных покупателей посчитаем как количество успешных сделок (у нас возможна только одна сделка на клиента за период)

Успешными сделки будем считать те, у кого Stage = Paymet Done и Month of Study больше 0.



   

In [3]:
ua=deals_df["Contact Name"].nunique()
print("UA:", ua)


filtered = deals_df.loc[
    (deals_df["Stage"] == "Payment Done") &
    (deals_df["Months of study"] > 0),
    "Offer Total Amount"
]

b = len(filtered)

print("B:", b)

UA: 17005
B: 837


У нас нет COGs, Revenue=GP. Но нужно только оплаченые месяцы.

In [4]:
r = deals_df.loc[
    (deals_df["Stage"] == "Payment Done") &
    (deals_df["Months of study"] > 0) &
    (deals_df["Course duration"] > 0),
].assign(
    Adjusted_Revenue=lambda df: df["Offer Total Amount"] * df["Months of study"] / df["Course duration"]
)["Adjusted_Revenue"].sum()

print("GP (Revenue, adjusted by study months):", round(r, 2))



GP (Revenue, adjusted by study months): 3314610.61


Количество транзакций это сумма всех Months of study.

In [5]:
t = deals_df["Months of study"].sum()
print("T:", t)


T: 4563.0


AC - вся сумма расходов на рекламу

In [6]:
ac = spend_df["Spend"].sum()
print("AC:", ac)


AC: 149523.45


Теперь можем расчитать все остальные метрики.

Средний чек тут посчитан просто как GP (Revenue) поделеная на количество транзакций. То, что был какойто первоначальный платеж, во внимание не принималось (то есть считала, что каждый месяц оплата за обучение одинакова). Пототому что мы не знаем, был ли еще один платеж в первый месяц обучения.

In [7]:
c=b/ua
aov=r/t
cm=r-ac
cltv=r/b
ltv=r/ua
cac =ac/b
ltc=ac/ua
apc=t/b




Теперь выведем все таблицей

In [8]:
# Заголовки и значения
headers = ["GP=Revenue", "UA", "B", "C1", "AOV",  "T",  "APC", "CLTV", "LTV", "AC", "CAC", "LTC", "CM"]
values  = [r, ua, b, c, aov,  t,  apc, cltv, ltv, ac, cac, ltc, cm]

line = "+" + "+".join(["-"*10 for _ in headers]) + "+"
print(line)
print("|" + "|".join(f"{h:^10}" for h in headers) + "|")
print(line)
print("|" + "|".join(f"{v:^10.2f}" if isinstance(v, (int, float)) else f"{v:^10}" for v in values) + "|")
print(line)



+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
|GP=Revenue|    UA    |    B     |    C1    |   AOV    |    T     |   APC    |   CLTV   |   LTV    |    AC    |   CAC    |   LTC    |    CM    |
+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+
|3314610.61| 17005.00 |  837.00  |   0.05   |  726.41  | 4563.00  |   5.45   | 3960.11  |  194.92  |149523.45 |  178.64  |   8.79   |3165087.16|
+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+


**Точка роста и гипотеза, исходя из общей юнит-экономики.**

Мы видим, что конверсия из потенциальных клиентов в покупатели в среднем составляет 5%. Будем считать это нашей точкой роста (бутылочное горлышко - низкая конверсия, увелича которую мы сможем значительно увеличить маржинальную прибыль).

*Гипотеза - добавив проверку правильности введенного номера телефона (когда клиент подает заявку), мы сможем увеличить конверсию до 7% (на 2%) за 2 недели.* Данная гипотеза базируется на одной из причин потерь заявок в Lost Reason.

**Как мы можем проверить эту гипотезу?**

Можем провести А/В тестирование.

Альтернативная гипотеза - если мы внедрим проверку телефона клиента в форме заявки (например чтоб человек ввел телефон дважды), это приведет к повышению конверсии до 7%, то есть на 40%  (до 7% +- 0,5%).

Нулевая гипотеза - конверсия не изменится.

Как мы будем проводить тестирование?
- Подготовить тестирование, все расчитать.
- Нам нужно разделить новых посетителей сайта на 2 репрезентативные группы. У одной группы будет версия сайта старая, у второй -  с обновленной формой подачи заявки (заявки на звонок менеджера, консультацию).
- Провести тест.
- Проанализировать результаты. Если результат конверсия группы с обновленной формой попадет в диапазон 7%+-0,5% - тогда принимаем гипотезу и внедряем изменения. Если нет - либо меняем эту гипотезу, либо вообще отбрасываем.







Можем ли мы провести такой тест за 2 недели?
-  Сначала нужно расчитать необходимое количество пользователей в каждой группе. По формуле (с достоверностью 95% и мощностью 80%) подсчитаем, что в каждой группе должно быть по  1900 пользователей. С учетом того, что мы не знаем кто из наших пользователей решит заполнить форму, мы должны проводить тест на просто новых посетителях. Конверсия из того кто перешел на сайт (а вернее кликнул) в посетителя, который заполнил форму равна 3,4% (примерно каждый 29й заполняет форму), значит нужно 1900 еще разделить на 0,034, получается 55882, а для двух групп это 111764.
- Потом посчитать сколько у нас в среднем есть посетителей сайта.
 Проведем подсчет:

In [9]:
total_clicks = spend_df["Clicks"].sum()
print("Сумма по колонке Clicks(за год):", total_clicks)


Сумма по колонке Clicks(за год): 498455


В год у нас 498455 переходов на сайт. Не принимая во внимание сезонность, в месяц это в среднем 41538 человек. Для проведения нашего теста нам понадобится около 2,69 месяца.

**Вывод:** за две недели провети такой тест (с указанными выше значениями достоверности и мощности) не получится.

Постараемся понять **дерево метрик**. Мы считали только метрики юнит-экономики (метрики принятия решений), поэтому на дереве будут только они. Кроме них у нас конечно есть атомные метрики, которые передавались в CRM автоматически.

Это перевернутое примерное дерево метрик, где целевая метрика (у нас из тех что есть это СМ) - внизу

                                UA
                                 |
                        +--------+--------+
                        |                 |
                       LTC               CAC
                                           |
                                           |
                                       B <--- C1 <--- UA
                                        |
                   +----------------+---+----------------+
                   |                |                    |
                 AOV              APC                   T
                   \                |                    /
                    \               +-------------------/
                     \                       |
                     +------ Revenue = B × AOV × APC -+
                                   |
                              +----+----+
                              |         |
                            CLTV        LTV
                                           |
                                          CM


В презентации будет примерное дерево метрик уже перевернутое, где целевая метрика - вверху.

**Теперь посчитаем юнит-экономику по продуктам. За весь период (год)**

Так как колонка Product заполнялась только в случае успешной сделки (и еще Waiting for Payment), то мы не можем посчитать UA и метрики связанные с UA. Маркетинговые компании в большинстве своем были тергетированы не по продуктам (таких было мало), поэтому из этой колонки данные получить мы тоже не можем. И AC мы также посчитать не можем. В презентации в разделе "Рекомендации и выводы" отмечу, что очень важно заполнять это поле сразу при создании сделки. Если человек еще не определился, так и указывать "не определился".

АС можно просто пропорционально как-то постараться посчитать, абстрактно, и вывести CM, но с таким же успехом в данном случае мы можем ориентироваться по показателю GP, который у нас равен Revenue.

In [11]:
pd.options.display.float_format = '{:,.2f}'.format


# Фильтруем только успешные сделки, где указаны оба поля
df_filtered = deals_df[
    (deals_df["Stage"] == "Payment Done") &
    (deals_df["Months of study"] > 0) &
    (deals_df["Course duration"] > 0)
].copy()

# Добавляем скорректированный доход
df_filtered["Adjusted_Revenue"] = (
    df_filtered["Offer Total Amount"] *
    df_filtered["Months of study"] /
    df_filtered["Course duration"]
)

# Группируем по Product
agg_df = df_filtered.groupby("Product", dropna=False).agg(
    Revenue=("Adjusted_Revenue", "sum"),
    T=("Months of study", "sum"),
    B=("Id", "count")
).reset_index()


# Добавляем расчетные поля
agg_df["AOV"] = (agg_df["Revenue"] / agg_df["T"]).round(2)  # Revenue / T
agg_df["APC"] = (agg_df["T"] / agg_df["B"]).round(2)       # T / B
agg_df["CLTV"] = (agg_df["Revenue"] / agg_df["B"]).round(2) # Revenue / B

# Вывод
agg_df


,Product,Revenue,T,B,AOV,APC,CLTV
0,Digital Marketing,"2,122,163.64","2,897.00",474,732.54,6.11,"4,477.14"
1,UX/UI Design,"854,863.64","1,161.00",226,736.32,5.14,"3,782.58"
2,Web Developer,"337,583.33",505.00,137,668.48,3.69,"2,464.11"


Тут показаны метрики, которые возможно расчитать по продуктам, по которым были продажи.

**Находим точку роста и формулируем гипотезу, исходя из юнит-экономики по продуктам.**

Мы видим, что CLTV по курсу Web Developer меньше в два раза, чем у двух других курсов. Проверив мы находим, что этот курс длится 6 месяцев, а два других - 11. Пусть это будет наша точка роста (узкое горлышко в том, что курс разработчика слишком короткий, и увеличение длительности курса может дать значительное изменение в прибыли).

*Гипотеза - увеличение длительности курса Web Developer до 11 месяцев даст увеличение Revenue по этому продукту на 83% в течение следующего года.*

**Как можем проверить эту гипотезу?** В этом случае мы не можем провести А/В тест. Нам нужно реально:
*   либо изменить длительность курса и потом через какое-то время проанализировать изменение (будут ли его меньше или больше покупать, как изменится CLTV)
*   либо запустить новый более длинный вараинт курса, оставив и старый. И потом сравнить их два.

Склоняюсь к второму варианту - так как он позволит сохранить количество новых клиентов, которые выбирают короткий курс. Но с учетом того, что внедрение нового курса займет ресурсы, как по мне стоит поступить так - указать на сайте, что есть еще более длительный курс Web Developer, но старт курса еше не известен - это поможет узнать, такой ли спрос.

Можно ли реализовать это за две недели?
- реальное внедрение нового курса - нет.
- проверка интереса к более длинному курсу - да, хотя лучше ее продлить на чуть дольше, чтоб было достовернее.




**Визуализация воронки (для презентации)**

Тут показана конверсия от предыдущего шага.

In [ ]:
import plotly.graph_objects as go

x_values = [498455, 17005 , 837]
y_labels = ["Total Clicks", "UA", "B"]

fig = go.Figure(go.Funnel(
    y=y_labels,
    x=x_values,
    textinfo="value+percent previous",
    hovertemplate="none"
))

fig.show()
